In [1]:
# There are 2 types of Agents - 
"""
1. Simple Agent:
    The agent enters a loop where it sends a request and checks whether the response contains a tool call.
    If so, it executes the function (e.g. get_weather) and appends the result to the conversation.
    The loop stops when there are no more tool calls and the response contains final text (output_text).
2. Objective-Based Agent:
    The agent is given a custom objective function (e.g. check whether the phrase "task complete" is in the output).
    It loops until the objective function returns True.
------------
get_capital(country) → calls a country API and returns Paris
get_weather(city) → calls a geocoding API to convert Paris → latitude/longitude, then calls the weather API using those coordinates.

"""
import os, sys
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
from llm_config import MODEL_GROQ, groq_api_key 
from getpass import getpass
import requests
import json
import requests
from dotenv import load_dotenv
load_dotenv(override=True)
capital_key = os.getenv("capital_key")

In [ ]:
from os import name
def get_capital(countrycode):
    countrycode = 'CA'
    response = requests.get(
        f"https://api.restcountries.com/countries/v5/codes.alpha_2/{countrycode}?pretty=1/",
        headers={
            "Authorization": capital_key
        }
    )
    data = response.json()
    return data['data']['objects'][0]['capitals'][0]['name'] 
    
capital_tool = {
    "type": "function",
    "name": "get_capital",
    "description": "Get the capital city of a country. Use this tool when the user asks for the weather in the capital of a country.",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "Name of the country"
            }
        },
        "required": ["country"],
        "additionalProperties": False
    },
    "strict": True
}
tools = [capital_tool, weather_tool]

In [46]:
get_capital("CA")

'Ottawa'

In [3]:
# Convert city → latitude/longitude
import requests
def get_weather(city):
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
    ) 
    geo_data = geo_response.json()
    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    weather_response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = weather_response.json()
    return data['current']['temperature_2m']

weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Name of the city"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    },
    "strict": True
}
tools = [capital_tool, weather_tool]

In [17]:
get_weather('Paris')

26.9

In [ ]:
# This agent sends a prompt (asking about the weather), then enters an agentic loop.
# At each turn, it calls the Responses API:

# 1. If the response contains a tool call: The agent executes the function (using our get_weather tool) and 
# appends the function result to the conversation as a new message.
# 2. If the response provides output text: The agent stops, printing the final output.
"""
User
 │
 │ "weather in capital of France"
 ▼
LLM
 │
 │ First needs capital
 ▼
get_capital(country="France")
 │
 ▼
"Paris"
 │
 ▼
LLM
 │
 │ Now needs Paris coordinates
 ▼
get_weather(latitude=48.8566, longitude=2.3522)
 │
 ▼
Weather
 │
 ▼
Final answer
"""
# from llm_config import groq, groq_url
from ast import arguments
from llm_config import ollama, MODEL_OLLAMA

messages = [
    {
        "role": "system",
        "content": """
        For weather questions asking about the capital of a country:
        1. First call get_capital to find the capital city.
        2. After receiving the capital, call get_weather using that city.
        3. Do not call get_weather before get_capital.
        """
    },
    {
        "role": "user",
        "content": "What's the weather in the capital of CA today?"
    }
]

while True:
    response = ollama.responses.create(
        model = MODEL_OLLAMA, 
        input = messages,
        tools = tools,
    )
 
    if response.output:
        for output_item in response.output:
            if output_item.type == "function_call":
                messages.append(output_item)
                if output_item.name == 'get_capital':
                    args = json.loads(output_item.arguments)
                    city = get_capital(args['country'])
                    messages.append(output_item)
                    messages.append({
                        "type": "function_call_output",
                        "call_id": output_item.call_id,
                        "output": str(city)})
                elif output_item.name == 'get_weather':
                    args = json.loads(output_item.arguments)
                    result = get_weather(args['city'])
                    messages.append(output_item)
                    messages.append({
                        "type": "function_call_output",
                        "call_id": output_item.call_id,
                        "output": str(result)
                    })
    if response.output_text:
        print("Final agent output:", response.output_text)
        break
            
